# Sprint 7C - Graph C GATv2 Explanation Runner

**Runner-only notebook.** Sprint 7C is analysis-only: it does not train models and does not change model, graph, loss, threshold, split, or feature artifacts. Analysis, plotting, and reporting logic stays in `scripts/analyze_sprint7c_graphc_gatv2_explanation.py`.

Execution plan: `docs/exec-plans/active/007c-sprint7c-graphc-gatv2-explanation.md`  
Runner boundary: `colab/README.md`

Before starting, confirm that the approved code revision is pushed and Drive contains the returned outputs from Sprint 5B, Sprint 6, Sprint 7, and Sprint 7B.

## Step 1 - Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Step 2 - Clone Or Update Repo Checkout

In [ ]:
%%bash
set -euo pipefail
REPO_URL="https://github.com/YasinEkici/crispr-gnn-offtarget.git"
REPO_DIR="/content/crispr-gnn-offtarget"
GIT_REF="sprint7/gat-gatv2"
if [ -d "$REPO_DIR/.git" ]; then
  cd "$REPO_DIR"
  git fetch origin "$GIT_REF"
  git checkout "$GIT_REF"
  git pull --ff-only origin "$GIT_REF"
else
  git clone --branch "$GIT_REF" "$REPO_URL" "$REPO_DIR"
  cd "$REPO_DIR"
fi
git rev-parse --short HEAD


## Step 3 - Dependency Sync And Runtime Check

In [ ]:
%%bash
set -euo pipefail
cd /content/crispr-gnn-offtarget
python -m pip install -q uv
uv sync
uv run python - <<'PY'
import pandas as pd
import sklearn
import matplotlib
print('pandas', pd.__version__)
print('sklearn', sklearn.__version__)
print('matplotlib', matplotlib.__version__)
PY


## Step 4 - Copy Required Sprint Output Artifacts From Drive

In [ ]:
%%bash
set -euo pipefail
cd /content/crispr-gnn-offtarget
DRIVE_ROOT_CANDIDATES=(
  "${DRIVE_ROOT:-/content/drive/MyDrive/crispr_gnn_offtarget}"
  "/content/drive/MyDrive/crispr-gnn-offtarget"
)
DRIVE_ROOT=""
for CANDIDATE in "${DRIVE_ROOT_CANDIDATES[@]}"; do
  if [ -d "$CANDIDATE" ]; then
    DRIVE_ROOT="$CANDIDATE"
    break
  fi
done
if [ -z "$DRIVE_ROOT" ]; then
  echo "No Drive project root found. Checked:" >&2
  printf '  %s
' "${DRIVE_ROOT_CANDIDATES[@]}" >&2
  find /content/drive/MyDrive -maxdepth 1 -type d | sort >&2
  exit 1
fi
echo "Using DRIVE_ROOT=$DRIVE_ROOT"
mkdir -p outputs
if [ -d "$DRIVE_ROOT/outputs" ]; then
  rsync -a "$DRIVE_ROOT/outputs/" outputs/
fi
copy_latest_returned() {
  local pattern="$1"
  local destination="$2"
  if [ -d "$destination" ]; then
    return 0
  fi
  local latest=""
  if [ -d "$DRIVE_ROOT/returned_outputs" ]; then
    latest=$(find "$DRIVE_ROOT/returned_outputs" -maxdepth 1 -type d -name "$pattern" | sort | tail -1 || true)
  fi
  if [ -n "$latest" ]; then
    mkdir -p "$destination"
    rsync -a "$latest/" "$destination/"
    echo "Copied $latest -> $destination"
  fi
}
copy_latest_returned 'sprint6_loss_comparison_*' outputs/sprint6/loss_comparison
copy_latest_returned 'sprint7_gat_gatv2_*' outputs/sprint7
copy_latest_returned 'sprint7b_gatv2_topology_*' outputs/sprint7b
copy_latest_returned 'sprint5b*' outputs/sprint5b
required=(
  outputs/sprint5b/graph_c/diagnostics/gcn_graph_c_predictions.csv
  outputs/sprint5b/graph_c/diagnostics/gcn_graph_c_fixed_threshold_metrics.csv
  outputs/sprint6/loss_comparison/diagnostics_sprint6/sprint6_loss_comparison_predictions.csv
  outputs/sprint6/loss_comparison/diagnostics_sprint6/imbalance_threshold_metrics.csv
  outputs/sprint7/gat_comparison.csv
  outputs/sprint7/diagnostics/gat_predictions.csv
  outputs/sprint7b/gatv2_topology_comparison.csv
  outputs/sprint7b/diagnostics/gatv2_topology_predictions.csv
  outputs/sprint7b/diagnostics/gatv2_topology_attention_summary.csv
)
missing=0
for path in "${required[@]}"; do
  if [ ! -f "$path" ]; then
    echo "Missing required Sprint 7C input: $path" >&2
    missing=1
  fi
done
if [ "$missing" -ne 0 ]; then
  echo "Available output artifacts:" >&2
  find outputs -maxdepth 5 -type f | sort | head -200 >&2 || true
  exit 1
fi
find outputs/sprint5b outputs/sprint6 outputs/sprint7 outputs/sprint7b -maxdepth 4 -type f | sort | head -120


## Step 5 - Run Sprint 7C Explanation Analysis

In [ ]:
%%bash
set -euo pipefail
cd /content/crispr-gnn-offtarget
RUN_ID="sprint7c_graphc_gatv2_explanation_$(date -u +%Y%m%d_%H%M%S)"
uv run python scripts/analyze_sprint7c_graphc_gatv2_explanation.py   --output-dir outputs/sprint7c
echo "$RUN_ID" > /content/sprint7c_graphc_gatv2_explanation_run_id.txt


## Step 6 - Copy Outputs Back To Drive

In [ ]:
%%bash
set -euo pipefail
cd /content/crispr-gnn-offtarget
DRIVE_ROOT="${DRIVE_ROOT:-/content/drive/MyDrive/crispr_gnn_offtarget}"
ALT_DRIVE_ROOT="/content/drive/MyDrive/crispr-gnn-offtarget"
if [ ! -d "$DRIVE_ROOT" ] && [ -d "$ALT_DRIVE_ROOT" ]; then
  DRIVE_ROOT="$ALT_DRIVE_ROOT"
fi
RUN_BASENAME=$(cat /content/sprint7c_graphc_gatv2_explanation_run_id.txt)
LOCAL_OUT="outputs/sprint7c"
RETURN_ROOT="$DRIVE_ROOT/returned_outputs/$RUN_BASENAME"
if [ -e "$RETURN_ROOT" ]; then
  echo "Output already exists in Drive: $RETURN_ROOT" >&2
  exit 1
fi
mkdir -p "$RETURN_ROOT"
rsync -a "$LOCAL_OUT/" "$RETURN_ROOT/"
find "$RETURN_ROOT" -maxdepth 3 -type f | sort | head -100


## Step 7 - Returned Artifact Checks

In [ ]:
%%bash
set -euo pipefail
cd /content/crispr-gnn-offtarget
OUT="outputs/sprint7c"
test -f "$OUT/sprint7c_identity_alignment_audit.md"
test -f "$OUT/sprint7c_graphc_gatv2_explanation_report.md"
test -f "$OUT/sprint7c_analysis_manifest.json"
test -f "$OUT/diagnostics/sprint7c_prediction_alignment_audit.csv"
test -f "$OUT/diagnostics/sprint7c_metric_recomputation.csv"
test -f "$OUT/diagnostics/sprint7c_error_transitions.csv"
test -f "$OUT/figures/sprint7c_error_transition_matrix.png"
test -f "$OUT/figures/sprint7c_attention_edge_kind_summary.png"
uv run python - <<'PY'
import json
from pathlib import Path
import pandas as pd
manifest = json.loads(Path('outputs/sprint7c/sprint7c_analysis_manifest.json').read_text())
if not manifest['contract']['analysis_only'] or not manifest['contract']['no_model_training']:
    raise SystemExit('Sprint 7C manifest contract drift')
metrics = pd.read_csv('outputs/sprint7c/diagnostics/sprint7c_metric_recomputation.csv')
view = metrics.loc[(metrics['analysis_model_id'].isin(['graph_c_gcn_s5b','graph_c_gatv2_s7b'])) & (metrics['split'] == 'test')]
print(view[['analysis_model_id','auprc','auroc','mcc','specificity','tn','fp','fn','tp']].to_string(index=False))
transitions = pd.read_csv('outputs/sprint7c/diagnostics/sprint7c_error_transitions.csv')
print(transitions['transition'].value_counts().to_string())
print('validated', Path('/content/sprint7c_graphc_gatv2_explanation_run_id.txt').read_text().strip())
PY
